In [ ]:
# 1. サンプルデータセットをダウンロード
%cd /content
!git clone https://github.com/wal-afk/drive_sim
%cd drive_sim
!git pull
!git restore .
!git clean -fd
%cd /content/drive_sim

!pip install -U plotly==6.9

In [ ]:
import yaml

from sim.drive_simulator import CarSim
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalLine, GoalCircle
from sim.drawer import SimDrawer, MissionDrawer
from sim.worlds.type_b_world import type_b_circuit
from sim.sign import Sign

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)

In [ ]:
class Mission7(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=40)
        # シミュレータでは「1つ目の標識のコース内側を通過 → 2つ目の標識の0.4m手前で停止」したら成功とする
        self.goals = [
            GoalCircle((1.6, 0.4), 0.15, should_stop=False),
            GoalCircle((4.07, 0.45), 0.15, should_stop=True),
        ]
        self.initial_xy = (0.2, 0.0)
        self.random_d_xy = (0.1, 0.1)
        self.random_d_yaw_deg = 5
        self.set_signs(
            [
                Sign(x=1.6, y=0.0, name="original"),
                Sign(x=4.25, y=0.81, name="warn"),
            ]
        )

    @staticmethod
    def command_func(alive, *, move, rotate, wait, search, auto, **kwargs):
        """回答例：
        ①autoで走行 ②1つ目の標識を半円の経路で回避 ③autoで走行 ④2つ目の標識の手前で停止
        """
        while alive():
            pos = search()
            if pos is None:
                auto(v=0.2)
            elif pos.name == "original":
                # ####### ここから下に「標識(original)が見つかった場合」のプログラムを書こう
                if pos.x > 0.4:
                    auto(v=0.2)
                elif pos.theta > 5:
                    rotate(w=45)
                elif pos.theta < -5:
                    rotate(w=-45)
                else:
                    # 左に90度回転 → 半径0.4mの右カーブで半円 → 左に90度回転
                    rotate(w=90, t=1)
                    wait()
                    move(v=0.2, r=-0.4, t=3.14 * 0.4 / 0.2)
                    wait()
                    rotate(w=90, t=1)
                    wait()
                ######## ここより上にプログラムを書こう
            else:
                # ####### ここから下に「original以外の標識が見つかった場合」のプログラムを書こう
                if pos.x > 0.4:
                    auto(v=0.2)
                else:
                    move(v=0)
                ######## ここより上にプログラムを書こう


sim = CarSim(prop, Mission7())
sim.run()
SimDrawer(sim).show()

In [ ]:
# 複数回実行して必ず成功するかのチェック
CHECK_CNT = 10
ok = 0
for i in range(CHECK_CNT):
    mission = Mission7()
    sim = CarSim(prop, mission)
    complete = sim.run()
    if complete:
        goal_cnt = sim.history.goal_cnt[-1]
        goals = len(mission.goals)
        goaled = goal_cnt == goals
        if goaled:
            ok += 1
        else:
            SimDrawer(sim).show()
            break
        print("-------------------------------OK:", ok)
